In [ ]:
# Equations entirely based off of Explosive Hazard Range spreadsheet from mentor. Sources needed.
# Do not use this code outside of CURE and the context of the CURE hybrid rocket motor program to ensure safety and compliance with regulations.

# Intended to find the safe distance from an explosive hazard based on N2O oxidizer and fuel mass in a hybrid rocket motor.
# The equations are based on the TNT equivalent of the explosive hazard, which is calculated using the mass of the oxidizer and fuel.
# The safe distance is then calculated using the TNT equivalent and a safety factor.

# Also, I definitely need to clean this up into a more readable and usable format, but it is 1AM and I am very eepy. May need a second or third set of eyes to double check eepy math to be safe :)

# N2O + solid to TNT RE factor: (m_ox_kg * 0.5) / (m_ox_kg + m_fuel_kg)
# N2O + liquid fuel to TNT RE factor: (m_ox_kg * 0.5 + m_fuel_kg) / (m_ox_kg + m_fuel_kg)
# LOX + solid to TNT RE factor: (m_ox_kg) / (m_ox_kg + m_fuel_kg)
# LOX + liquid fuel to TNT RE factor: (m_ox_kg + m_fuel_kg) / (m_ox_kg + m_fuel_kg) = 1
# black powder to TNT RE factor: 0.55 
# ^ for ejection charge standoff distance: if we use 10g BP, that's 0.0055kg TNT. If we want to find a safe distance for any ejection charge testing under 10g, we can input 0.0055kg TNT into the tnt_eq_kg_override

# CURE Values TNT eq:
# 54mm = 0.52kg
# 75mm = 1.29kg
# Short 5" = 2.62kg
# Long 5" = 3.84kg
# 152mm = 6.04kg

# the ultimate override for the standoff distances. use this if you already know tnt equivalents and want to find quick standoffs.
tnt_eq_kg_override = 0.0055

# the next layer of override. if you have an accurate value for oxidizer and fuel mass, you can use these to calculate tnt equivalent and standoff distances.
m_ox_kg_override = 0
m_fuel_kg_override = 0

# if you do not have any of the above information, the code will calculate the oxidizer mass based on the volume of the oxidizer tank and the density of N2O at 53F. The fuel mass will be calculated using an oxidizer-to-fuel mass ratio
v_ox = 0.0075  # m^3 (7500cc). Volume of oxidizer tank to fall back on when no overrides are used. Part of calculating m_ox_kg
rho_ox = 842.1  # 842.1 kg/m^3. n2o density at 53F: reference lookup table (coldest operating temperature = most n2o mass in a worst case scenario = largest blast with available volume). Part of calculating m_ox_kg. 93F=621.5kg/m^3 for reference for best case, changes 1.3x ear by ~40ft
ox_f_ratio = 6 # oxidizer-to-fuel mass ratio. will automatically determine m_fuel_kg using this ratio if m_ox_kg_override is 0. 5-6 is typical for N2O and ABS

m_ox_kg_calculated = v_ox * rho_ox # mass = density * volume ; kg = kg/m^3 * m^3

m_ox_kg = m_ox_kg_override if m_ox_kg_override > 0 else m_ox_kg_calculated
if m_ox_kg_override <= 0:
    print(f"Using calculated oxidizer mass: {m_ox_kg_calculated:.2f} kg")
else:
    print(f"Using overridden oxidizer mass: {m_ox_kg_override:.2f} kg")

m_fuel_kg_calculated = m_ox_kg / ox_f_ratio

m_fuel_kg = m_fuel_kg_override if m_fuel_kg_override > 0 else m_fuel_kg_calculated
if m_ox_kg_override <= 0:
    print(f"Using calculated fuel mass: {m_fuel_kg_calculated:.2f} kg")
else:
    print(f"Using overridden fuel mass: {m_fuel_kg_override:.2f} kg")

impulse = 200 * (m_ox_kg + m_fuel_kg) * 9.81 # Impulse = specific impulse * total mass * gravity. Specific impulse of 200s approximation
print(f"Estimated Impulse (from masses, not TNT eq): {impulse:.2f} N·s")
# print letter classification for funsies, list stored as {Letter, max_impulse_for_classification}
classification = [["A",2.5], ["B",5], ["C",10], ["D",20], ["E",40], ["F",80], ["G",160], ["H",320], ["I",640], ["J",1280], ["K",2560], ["L",5120], ["M",10240], ["N",20480], ["O",40960], ["P",81920], ["Q",163840], ["R",327680], ["S",655360]]
# loop impulse through classification until impulse is less than max_impulse, then print letter classification
for letter, max_impulse in classification:
    if impulse < max_impulse:
        print(f"Estimated Impulse Classification: {letter}")
        break

# RE factor relative to TNT (i.e. TNT = 1.00).
re_factor = (m_ox_kg * 0.5) / (m_ox_kg + m_fuel_kg)  

tnt_eq_kg_calculated = re_factor * (m_ox_kg + m_fuel_kg)

tnt_eq_kg = tnt_eq_kg_override if tnt_eq_kg_override > 0 else tnt_eq_kg_calculated
if tnt_eq_kg_override > 0:
    print(f"\n(Above data about impulse and masses may not be related to the TNT equivalent if tnt_eq_kg_override is used.)")
    print(f"\nUsing overridden TNT equivalent: {tnt_eq_kg_override:.2f} kg TNT") 
else:
    print(f"Using calculated TNT equivalent: {tnt_eq_kg_calculated:.2f} kg TNT")

cube_root_tnt_eq_kg = tnt_eq_kg ** (1/3)

# would like to know where these equations come from - question for mentor. 
# I know standoff pressures are based on a national standard for blast hazards, but deriving the shockwave effects and distances are a mystery to me. 
# They may just be constants found through trial and error research papers? Seems like some are of similar format, and that they are all some form of fitted equation for scaled distance as a function of pressure. 
# Some of the equations produced larger radii at higher pressures (and some at lower), so I chose the functions that created larger radii at each pressure to be conservative.
calculated_ear_damage_0_1_psi = cube_root_tnt_eq_kg * ((191/0.1)**(1/1.38))
light_blast_damage_1_psi = cube_root_tnt_eq_kg * ((725/1)**(0.6098)) # leaving the 1 for 1psi visibility in the equation. still would love to know where these equations and constants come from
moderate_blast_damage_5_psi = cube_root_tnt_eq_kg * ((725/5)**(0.6098))
heavy_blast_damage_20_psi = cube_root_tnt_eq_kg * 31.9* (20**(-0.433)) + 7.09
extreme_damage_200_psi = cube_root_tnt_eq_kg * 31.9* (200**(-0.433)) + 7.09
complete_destruction_3000_psi = cube_root_tnt_eq_kg * 31.9* (3000**(-0.433)) + 7.09

print("\nCalculated Safe Distances:")
print(f"Calculated Ear Damage (0.1 psi): {calculated_ear_damage_0_1_psi:.2f} ft")
print(f"Light Blast Damage (1 psi): {light_blast_damage_1_psi:.2f} ft")
print(f"Moderate Blast Damage (5 psi): {moderate_blast_damage_5_psi:.2f} ft")
print(f"Heavy Blast Damage (20 psi): {heavy_blast_damage_20_psi:.2f} ft")
print(f"Extreme Damage (200 psi): {extreme_damage_200_psi:.2f} ft")
print(f"Complete Destruction (3000 psi): {complete_destruction_3000_psi:.2f} ft")

# Factor of safety to take into account for uncertanties in calculation such that a variance of a few feet in modeling the hazard is not a safety concern 
# Factor of safety 1.3x:
print("\nFactor of Safety 1.3x:")
print(f"Calculated Ear Damage (0.1 psi): {calculated_ear_damage_0_1_psi * 1.3:.2f} ft")
print(f"Light Blast Damage (1 psi): {light_blast_damage_1_psi * 1.3:.2f} ft")
print(f"Moderate Blast Damage (5 psi): {moderate_blast_damage_5_psi * 1.3:.2f} ft")
print(f"Heavy Blast Damage (20 psi): {heavy_blast_damage_20_psi * 1.3:.2f} ft")
print(f"Extreme Damage (200 psi): {extreme_damage_200_psi * 1.3:.2f} ft")
print(f"Complete Destruction (3000 psi): {complete_destruction_3000_psi * 1.3:.2f} ft")

Using calculated oxidizer mass: 6.32 kg
Using calculated fuel mass: 1.05 kg
Estimated Impulse (from masses, not TNT eq): 14456.75 N·s
Estimated Impulse Classification: N

(Above data about impulse and masses may not be related to the TNT equivalent if tnt_eq_kg_override is used.)

Using overridden TNT equivalent: 0.01 kg TNT

Calculated Safe Distances:
Calculated Ear Damage (0.1 psi): 42.11 ft
Light Blast Damage (1 psi): 9.80 ft
Moderate Blast Damage (5 psi): 3.67 ft
Heavy Blast Damage (20 psi): 8.63 ft
Extreme Damage (200 psi): 7.66 ft
Complete Destruction (3000 psi): 7.27 ft

Factor of Safety 1.3x:
Calculated Ear Damage (0.1 psi): 54.74 ft
Light Blast Damage (1 psi): 12.73 ft
Moderate Blast Damage (5 psi): 4.77 ft
Heavy Blast Damage (20 psi): 11.22 ft
Extreme Damage (200 psi): 9.96 ft
Complete Destruction (3000 psi): 9.45 ft
